# Análise de gastos do cartão corporativo do governo brasileiro

## 1. Limpeza dos dados
    Remoção de **linhas duplicadas**
    Os valores nulos correspondem a dados sigilosos e foram mantidos.
    Valores financeiros: convertidos de string para float.

## 2. Agrupamento de gastos por órgão

- Fonte: Portal da Transparência: https://portaldatransparencia.gov.br/download-de-dados/cpgf
- Período: 3 meses [Junho, Julho e Agosto]

In [1]:
import pandas as db
from pathlib import Path
import numpy as np

## Transformar os 3 arquivos .csv em um único dataframe utilizando listas de compreensão
    - Pathlib utilizado para que funcione tanto em windows e *nix

In [2]:
pasta = Path("dados")
meses= ["202606","202607","202608"]
dados = db.concat([db.read_csv(pasta/f"{mes}_CPGF.csv",sep=';',encoding='latin-1',index_col=False)for mes in meses], ignore_index=True)

## Remoção de duplicados:
    Resolvi remover, pois, não há informação clara sobre dados repetidos.

In [3]:
print(f"Quantidade de dados repetidos: {dados.duplicated().sum()}")
dados = dados.drop_duplicates(keep='first').reset_index(drop=True)
dados.head()

Quantidade de dados repetidos: 11451


,CÓDIGO ÓRGÃO SUPERIOR,NOME ÓRGÃO SUPERIOR,CÓDIGO ÓRGÃO,NOME ÓRGÃO,CÓDIGO UNIDADE GESTORA,NOME UNIDADE GESTORA,ANO EXTRATO,MÊS EXTRATO,CPF PORTADOR,NOME PORTADOR,CNPJ OU CPF FAVORECIDO,NOME FAVORECIDO,TRANSAÇÃO,DATA TRANSAÇÃO,VALOR TRANSAÇÃO
0,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.195.852-**,JOSE UELISSON ALVES LEITE,27698586000166,CASA DOS FILTROS RONDONIA IMPORTACAO E EXPORTA...,COMPRA A/V - R$ - APRES,12/05/2026,348
1,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.562.861-**,ANTONIO CARLOS MELO DOS SANTOS,13018525000104,ADAIL PEREIRA ALVES DA SILVA 25837451153,COMPRA A/V - R$ - APRES,15/05/2026,1600
2,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.195.852-**,JOSE UELISSON ALVES LEITE,53254623000157,PH ELETRICIDADE E SEGURANCA LTDA,COMPRA A/V - R$ - APRES,29/04/2026,450
3,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.195.852-**,JOSE UELISSON ALVES LEITE,53254623000157,PH ELETRICIDADE E SEGURANCA LTDA,COMPRA A/V - R$ - APRES,29/04/2026,"1594,78"
4,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.609.911-**,ISABELA RODRIGUES TEIXEIRA,1373250000130,GOIAS PARAFUSOS LTDA,COMPRA A/V - R$ - APRES,13/05/2026,"250,5"


## Transformar o campo 'VALOR TRANSAÇÃO' em ponto flutuante:
    - Em R$ o ponto separa os milhares e a virgula os centavos;
    - O float só utiliza ponto para centavos.
    - Temos que retirar os pontos; Substituir a virgula por ponto; e transforma em ponto flutuante

In [4]:
dados['VALOR TRANSAÇÃO'] = ( 
    dados['VALOR TRANSAÇÃO']
    .astype(str)
    .str.replace('.','')
    .str.replace(',','.')
    .astype(float)
    .round(2)
)

dados.head()

,CÓDIGO ÓRGÃO SUPERIOR,NOME ÓRGÃO SUPERIOR,CÓDIGO ÓRGÃO,NOME ÓRGÃO,CÓDIGO UNIDADE GESTORA,NOME UNIDADE GESTORA,ANO EXTRATO,MÊS EXTRATO,CPF PORTADOR,NOME PORTADOR,CNPJ OU CPF FAVORECIDO,NOME FAVORECIDO,TRANSAÇÃO,DATA TRANSAÇÃO,VALOR TRANSAÇÃO
0,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.195.852-**,JOSE UELISSON ALVES LEITE,27698586000166,CASA DOS FILTROS RONDONIA IMPORTACAO E EXPORTA...,COMPRA A/V - R$ - APRES,12/05/2026,348.00
1,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.562.861-**,ANTONIO CARLOS MELO DOS SANTOS,13018525000104,ADAIL PEREIRA ALVES DA SILVA 25837451153,COMPRA A/V - R$ - APRES,15/05/2026,1600.00
2,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.195.852-**,JOSE UELISSON ALVES LEITE,53254623000157,PH ELETRICIDADE E SEGURANCA LTDA,COMPRA A/V - R$ - APRES,29/04/2026,450.00
3,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.195.852-**,JOSE UELISSON ALVES LEITE,53254623000157,PH ELETRICIDADE E SEGURANCA LTDA,COMPRA A/V - R$ - APRES,29/04/2026,1594.78
4,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2026,6,***.609.911-**,ISABELA RODRIGUES TEIXEIRA,1373250000130,GOIAS PARAFUSOS LTDA,COMPRA A/V - R$ - APRES,13/05/2026,250.50


## Criar uma tabela "pivoteada" para exibir os gastos do mês em coluna e "enxugar" para deixar somente os seguintes dados:
    - NOME ÓRGÃO SUPERIOR
    - GASTO EM JUNHO
    - GASTO EM JULHO
    - VARIAÇÃO % JUN/JUL
    - GASTO EM AGOSTO
    - VARIAÇÃO % JUL/AGO 
    - TOTAL GASTO NO TRIMESTRE

In [13]:
gastos_total = dados.pivot_table(index='NOME ÓRGÃO SUPERIOR',columns='MÊS EXTRATO',values='VALOR TRANSAÇÃO',aggfunc='sum',fill_value=0).reset_index()
gastos_total['VARIAÇÃO % JUN/JUL'] = ((gastos_total[7] - gastos_total[6]) / gastos_total[6].replace(0, np.nan).astype(float) ).fillna(0).apply(lambda x: f"{x:.2%}")
gastos_total['VARIAÇÃO % JUL/AGO'] = ((gastos_total[8] - gastos_total[7]) / gastos_total[7].replace(0, np.nan).astype(float) ).fillna(0).apply(lambda x: f"{x:.2%}")
gastos_total = gastos_total.reindex(columns=['NOME ÓRGÃO SUPERIOR',6,7,'VARIAÇÃO % JUN/JUL',8, 'VARIAÇÃO % JUL/AGO'])
gastos_total.columns = ['NOME ÓRGÃO SUPERIOR','TOTAL GASTO EM JUN','TOTAL GASTO EM JUL','VARIAÇÃO % JUN/JUL','TOTAL GASTO EM AGO', 'VARIAÇÃO % JUL/AGO']
gastos_total['TOTAL GASTO NO TRIMESTRE'] = gastos_total['TOTAL GASTO EM JUN'] + gastos_total['TOTAL GASTO EM JUL'] + gastos_total['TOTAL GASTO EM AGO']
gastos_total.sort_values(by='TOTAL GASTO NO TRIMESTRE', ascending=False).reset_index(drop=True)

,NOME ÓRGÃO SUPERIOR,TOTAL GASTO EM JUN,TOTAL GASTO EM JUL,VARIAÇÃO % JUN/JUL,TOTAL GASTO EM AGO,VARIAÇÃO % JUL/AGO,TOTAL GASTO NO TRIMESTRE
0,Ministério da Defesa,1756212.09,1662152.75,-5.36%,1809639.92,8.87%,5228004.76
1,Ministério da Educação,1848245.91,1392989.17,-24.63%,1379008.97,-1.00%,4620244.05
2,Presidência da República,997994.52,1007560.06,0.96%,1275054.77,26.55%,3280609.35
3,Ministério da Justiça e Segurança Pública,1101488.78,888804.84,-19.31%,958747.28,7.87%,2949040.90
4,Ministério do Planejamento e Orçamento,750395.07,662614.59,-11.70%,801015.43,20.89%,2214025.09
5,Ministério do Meio Ambiente e Mudança do Cl,411651.27,408558.04,-0.75%,430960.10,5.48%,1251169.41
6,Ministério do Desenvolvimento Agrário e Agr,370410.66,317333.43,-14.33%,277358.47,-12.60%,965102.56
7,Ministério da Saúde,199720.24,205991.66,3.14%,259768.34,26.11%,665480.24
8,Ministério da Agricultura e Pecuária,184921.34,165380.49,-10.57%,194663.44,17.71%,544965.27
9,Ministério da Fazenda,216252.04,157529.39,-27.15%,152509.29,-3.19%,526290.72


SyntaxError: invalid syntax (1055880104.py, line 1)